In [1]:
# ============================================================
# 삼성전자 주가 머신러닝 예측
#
# 목표
# 1. 삼성전자 일봉 다운로드
# 2. 기술적 지표 생성
# 3. 다음 거래일 수익률 예측
# 4. RandomForest 모델 학습
# 5. 테스트 데이터 평가
# 6. 다음 거래일 예상 종가 계산
# 7. Feature Importance 확인
#
# 종목:
# 삼성전자 005930.KS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ============================================================
# 0. 환경 설정
# ============================================================

TICKER = "005930.KS"
START_DATE = "2015-01-01"

TEST_RATIO = 0.20
RANDOM_STATE = 42


# ------------------------------------------------------------
# 한글 폰트 설정
# ------------------------------------------------------------

system = platform.system()

if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"

elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"

else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 삼성전자 데이터 다운로드
# ============================================================

df = yf.download(
    TICKER,
    start=START_DATE,
    auto_adjust=False,
    progress=False,
)


# ------------------------------------------------------------
# yfinance 버전에 따라 MultiIndex가 반환되는 경우 처리
# ------------------------------------------------------------

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)


df = df[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ]
].copy()


df = df.dropna()


print("데이터 기간")
print(df.index.min(), "~", df.index.max())

print()
print("데이터 크기:", df.shape)

display(df.tail())


# ============================================================
# 2. 기술적 지표 함수
# ============================================================

def calculate_rsi(series, period=14):

    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period,
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period,
    ).mean()

    rs = avg_gain / (avg_loss + 1e-10)

    return 100 - (100 / (1 + rs))


def calculate_atr(data, period=14):

    prev_close = data["Close"].shift(1)

    tr1 = data["High"] - data["Low"]

    tr2 = (
        data["High"]
        - prev_close
    ).abs()

    tr3 = (
        data["Low"]
        - prev_close
    ).abs()

    true_range = pd.concat(
        [
            tr1,
            tr2,
            tr3,
        ],
        axis=1,
    ).max(axis=1)

    return true_range.rolling(period).mean()


# ============================================================
# 3. Feature Engineering
# ============================================================

data = df.copy()


# ------------------------------------------------------------
# 수익률
# ------------------------------------------------------------

data["Return1"] = (
    data["Close"]
    .pct_change(1)
)

data["Return5"] = (
    data["Close"]
    .pct_change(5)
)

data["Return10"] = (
    data["Close"]
    .pct_change(10)
)

data["Return20"] = (
    data["Close"]
    .pct_change(20)
)


# ============================================================
# 이동평균
# ============================================================

for period in [
    5,
    20,
    60,
    120,
]:

    data[f"MA{period}"] = (
        data["Close"]
        .rolling(period)
        .mean()
    )

    # 현재 가격이 이동평균에서 얼마나 떨어져 있는지
    data[f"MA{period}Ratio"] = (
        data["Close"]
        / data[f"MA{period}"]
        - 1
    )


# ============================================================
# 이동평균 기울기
# ============================================================

data["MA5Slope"] = (
    data["MA5"]
    .pct_change(5)
)

data["MA20Slope"] = (
    data["MA20"]
    .pct_change(5)
)

data["MA60Slope"] = (
    data["MA60"]
    .pct_change(5)
)


# ============================================================
# 변동성
# ============================================================

data["Volatility5"] = (
    data["Return1"]
    .rolling(5)
    .std()
)

data["Volatility20"] = (
    data["Return1"]
    .rolling(20)
    .std()
)


# ============================================================
# RSI
# ============================================================

data["RSI14"] = calculate_rsi(
    data["Close"],
    period=14,
)


# ============================================================
# ATR
# ============================================================

data["ATR14"] = calculate_atr(
    data,
    period=14,
)

data["ATR14Pct"] = (
    data["ATR14"]
    / data["Close"]
)


# ============================================================
# 거래량
# ============================================================

data["VolumeMA20"] = (
    data["Volume"]
    .rolling(20)
    .mean()
)

data["VolumeRatio20"] = (
    data["Volume"]
    / data["VolumeMA20"]
)


data["VolumeChange"] = (
    data["Volume"]
    .pct_change()
)


# 너무 큰 이상치 제거
data["VolumeChange"] = (
    data["VolumeChange"]
    .clip(
        lower=-5,
        upper=5,
    )
)


# ============================================================
# 장중 가격 변동
# ============================================================

data["HighLowPct"] = (
    data["High"]
    - data["Low"]
) / data["Close"]


# ============================================================
# 시가 갭
# ============================================================

data["OpenGapPct"] = (
    data["Open"]
    / data["Close"].shift(1)
    - 1
)


# ============================================================
# Momentum
# ============================================================

data["Momentum5"] = (
    data["Close"]
    / data["Close"].shift(5)
    - 1
)

data["Momentum20"] = (
    data["Close"]
    / data["Close"].shift(20)
    - 1
)


# ============================================================
# MACD
# ============================================================

ema12 = (
    data["Close"]
    .ewm(
        span=12,
        adjust=False,
    )
    .mean()
)

ema26 = (
    data["Close"]
    .ewm(
        span=26,
        adjust=False,
    )
    .mean()
)

macd = ema12 - ema26

signal = (
    macd
    .ewm(
        span=9,
        adjust=False,
    )
    .mean()
)


data["MACDPct"] = (
    macd
    / data["Close"]
)

data["MACDSignalPct"] = (
    signal
    / data["Close"]
)


# ============================================================
# Bollinger Band 위치
# ============================================================

std20 = (
    data["Close"]
    .rolling(20)
    .std()
)

data["BollingerZ"] = (
    data["Close"]
    - data["MA20"]
) / (std20 + 1e-10)


# ============================================================
# 4. Target 생성
# ============================================================
#
# 오늘 종가 기준
#
# 다음 거래일 수익률 =
#
#     내일 종가 / 오늘 종가 - 1
#
# ============================================================

data["TargetReturn"] = (
    data["Close"]
    .shift(-1)
    / data["Close"]
    - 1
)


data["TargetClose"] = (
    data["Close"]
    .shift(-1)
)


# ============================================================
# 5. 사용할 Feature
# ============================================================

FEATURES = [

    # 수익률
    "Return1",
    "Return5",
    "Return10",
    "Return20",

    # 이동평균 거리
    "MA5Ratio",
    "MA20Ratio",
    "MA60Ratio",
    "MA120Ratio",

    # 이동평균 기울기
    "MA5Slope",
    "MA20Slope",
    "MA60Slope",

    # 변동성
    "Volatility5",
    "Volatility20",

    # 기술적 지표
    "RSI14",
    "ATR14Pct",

    # 거래량
    "VolumeRatio20",
    "VolumeChange",

    # 가격 움직임
    "HighLowPct",
    "OpenGapPct",

    # 모멘텀
    "Momentum5",
    "Momentum20",

    # MACD
    "MACDPct",
    "MACDSignalPct",

    # Bollinger
    "BollingerZ",
]


# ============================================================
# 6. 머신러닝 데이터 생성
# ============================================================

model_data = data[
    FEATURES
    + [
        "Close",
        "TargetReturn",
        "TargetClose",
    ]
].copy()


model_data = (
    model_data
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna()
)


print("머신러닝 데이터")

display(
    model_data
    .tail()
)


# ============================================================
# 7. Train / Test 분리
# ============================================================
#
# 중요:
#
# 주식 데이터에서는 random shuffle을 사용하지 않는다.
#
# 앞 80% = Train
# 뒤 20% = Test
#
# ============================================================

split_index = int(
    len(model_data)
    * (1 - TEST_RATIO)
)


train = (
    model_data
    .iloc[:split_index]
    .copy()
)

test = (
    model_data
    .iloc[split_index:]
    .copy()
)


X_train = train[FEATURES]

y_train = train["TargetReturn"]


X_test = test[FEATURES]

y_test = test["TargetReturn"]


print(
    "Train:",
    X_train.shape,
)

print(
    "Test:",
    X_test.shape,
)


print()

print(
    "Train 기간:",
    train.index.min(),
    "~",
    train.index.max(),
)

print(
    "Test 기간:",
    test.index.min(),
    "~",
    test.index.max(),
)


# ============================================================
# 8. Random Forest
# ============================================================

model = RandomForestRegressor(

    n_estimators=500,

    max_depth=8,

    min_samples_leaf=5,

    max_features=0.8,

    random_state=RANDOM_STATE,

    n_jobs=-1,
)


model.fit(
    X_train,
    y_train,
)


# ============================================================
# 9. 예측
# ============================================================

pred_return = model.predict(
    X_test
)


# ------------------------------------------------------------
# 수익률 → 종가
# ------------------------------------------------------------

pred_close = (
    test["Close"].values
    * (1 + pred_return)
)


actual_close = (
    test["TargetClose"]
    .values
)


# ============================================================
# 10. 모델 평가
# ============================================================

return_mae = mean_absolute_error(
    y_test,
    pred_return,
)

return_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_return,
    )
)


price_mae = mean_absolute_error(
    actual_close,
    pred_close,
)

price_rmse = np.sqrt(
    mean_squared_error(
        actual_close,
        pred_close,
    )
)


r2 = r2_score(
    y_test,
    pred_return,
)


# ------------------------------------------------------------
# 상승 / 하락 방향 정확도
# ------------------------------------------------------------

actual_direction = (
    y_test.values > 0
)

pred_direction = (
    pred_return > 0
)


direction_accuracy = (
    actual_direction
    == pred_direction
).mean()


print("=" * 60)

print("모델 평가")

print("=" * 60)

print(
    f"수익률 MAE       : "
    f"{return_mae * 100:.3f}%"
)

print(
    f"수익률 RMSE      : "
    f"{return_rmse * 100:.3f}%"
)

print(
    f"종가 MAE         : "
    f"{price_mae:,.0f}원"
)

print(
    f"종가 RMSE        : "
    f"{price_rmse:,.0f}원"
)

print(
    f"R²               : "
    f"{r2:.4f}"
)

print(
    f"상승/하락 정확도 : "
    f"{direction_accuracy * 100:.2f}%"
)


# ============================================================
# 11. 예측 결과 DataFrame
# ============================================================

result = pd.DataFrame(
    index=test.index,
)


result["현재종가"] = (
    test["Close"]
)

result["실제다음종가"] = (
    test["TargetClose"]
)

result["예측다음종가"] = (
    pred_close
)

result["실제수익률(%)"] = (
    y_test.values
    * 100
)

result["예측수익률(%)"] = (
    pred_return
    * 100
)

result["실제방향"] = np.where(
    result["실제수익률(%)"] > 0,
    "상승",
    "하락",
)

result["예측방향"] = np.where(
    result["예측수익률(%)"] > 0,
    "상승",
    "하락",
)

result["방향적중"] = (
    result["실제방향"]
    == result["예측방향"]
)


display(
    result
    .tail(20)
    .round(2)
)


# ============================================================
# 12. 실제 가격 vs 예측 가격
# ============================================================

plt.figure(
    figsize=(15, 6)
)


plt.plot(
    result.index,
    result["실제다음종가"],
    label="실제 다음 종가",
)

plt.plot(
    result.index,
    result["예측다음종가"],
    label="예측 다음 종가",
)


plt.title(
    "삼성전자 다음 거래일 종가 예측"
)

plt.xlabel(
    "예측 기준일"
)

plt.ylabel(
    "가격(원)"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 13. Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature":
        FEATURES,

    "Importance":
        model.feature_importances_,
})


importance = (
    importance
    .sort_values(
        "Importance",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    importance
)


plt.figure(
    figsize=(10, 7)
)


top_features = (
    importance
    .head(15)
    .sort_values(
        "Importance"
    )
)


plt.barh(
    top_features["Feature"],
    top_features["Importance"],
)


plt.title(
    "Feature Importance"
)

plt.xlabel(
    "Importance"
)

plt.tight_layout()

plt.show()


# ============================================================
# 14. 전체 데이터로 최종 모델 재학습
# ============================================================

final_model = RandomForestRegressor(

    n_estimators=500,

    max_depth=8,

    min_samples_leaf=5,

    max_features=0.8,

    random_state=RANDOM_STATE,

    n_jobs=-1,
)


final_model.fit(
    model_data[FEATURES],
    model_data["TargetReturn"],
)


# ============================================================
# 15. 가장 최근 데이터로 다음 거래일 예측
# ============================================================
#
# Target이 없어도 되므로
# 원래 data에서 가장 최근 Feature 데이터 사용
#
# ============================================================

latest_features = (
    data[FEATURES]
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna()
    .iloc[[-1]]
)


latest_date = (
    latest_features.index[-1]
)


latest_close = (
    data.loc[
        latest_date,
        "Close",
    ]
)


next_return = (
    final_model
    .predict(
        latest_features
    )[0]
)


next_close = (
    latest_close
    * (1 + next_return)
)


direction = (
    "상승"
    if next_return > 0
    else "하락"
)


print()
print("=" * 60)

print("삼성전자 다음 거래일 예측")

print("=" * 60)

print(
    f"기준일        : "
    f"{latest_date:%Y-%m-%d}"
)

print(
    f"현재 종가     : "
    f"{latest_close:,.0f}원"
)

print(
    f"예측 수익률   : "
    f"{next_return * 100:+.2f}%"
)

print(
    f"예측 방향     : "
    f"{direction}"
)

print(
    f"예상 다음종가 : "
    f"{next_close:,.0f}원"
)

print("=" * 60)

ModuleNotFoundError: No module named 'yfinance'